# PoliMillionaire Game Tests

Run live games with selected local models, one after another.

No generated-answer API is used.


## Setup

Find the repo and local model cache.


In [5]:
import gc
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault(
    "HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub")
)

print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


Repo: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire
HF cache: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/hf_home


## Settings

Pick models here. Each selected model gets its own game run.


In [6]:
RUN_API_CHECK = True
RUN_LIVE_GAME = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_ID = 0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen 2.5 3B", "model_id": "Qwen/Qwen2.5-3B-Instruct", "run": False},
    {"label": "Qwen 2.5 7B", "model_id": "Qwen/Qwen2.5-7B-Instruct", "run": True},
]

SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

print("API check:", RUN_API_CHECK)
print("Live game:", RUN_LIVE_GAME)
print("Competition:", COMPETITION_ID)
print("Models:")
for item in MODELS_TO_RUN:
    print("-", item["label"], item["model_id"], "run=", item["run"])


API check: True
Live game: True
Competition: 0
Models:
- Gemma 4 E2B google/gemma-4-E2B-it run= False
- Gemma 4 E4B google/gemma-4-E4B-it run= False
- Qwen 2.5 3B Qwen/Qwen2.5-3B-Instruct run= False
- Qwen 2.5 7B Qwen/Qwen2.5-7B-Instruct run= True


## Login

Use environment variables, Colab secrets, or prompt mode.


In [7]:
import getpass

from dotenv import load_dotenv

from millionaire_client import AuthenticationError, MillionaireClient

# Load environment variables from .env file
load_dotenv()

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")

try:
    from google.colab import userdata

    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass

if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")

print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))


Username configured: True
Password configured: True


## Competitions

Turn on `RUN_API_CHECK` to list games.


In [8]:
client = None

if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")


Logged in as promptPotamo
0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


## Run Selected Models

Each model warms up, plays if enabled, then unloads.


In [9]:
# !pip install  -qU colab-xterm
# %load_ext colabxterm
# # It may ask you to restart, accept and run this cell again

### For Colab: Run the following commands on the terminal below:
To spawn the server deamon please run the cell with ```%xterm```. It will create a concurrent shell where we can paste the following commands but, at the same time, also run other cells!

```sudo apt-get install zstd```

```curl -fsSL https://ollama.com/install.sh | sh```

```ollama serve & ollama pull llama3```

In [10]:
# %xterm

In [ ]:
import logging
import sys
from typing import Any

from src.polimillionaire.types import AnswerOption, Question

root = logging.getLogger()
if not any(isinstance(h, logging.StreamHandler) for h in root.handlers):
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(
        logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s")
    )
    root.addHandler(handler)
root.setLevel(logging.INFO)


In [14]:
"""
RAG Web Retrieval Pipeline
==========================
Improvements over baseline:
  1. Async fetching        — concurrent HTTP with httpx + semaphore
  2. Hybrid retrieval      — BM25 + dense embeddings fused with RRF
  3. Cross-encoder rerank  — sentence-transformers cross-encoder
  4. Query expansion       — LLM rewrites query into N variants before search
"""

from __future__ import annotations

import asyncio
import hashlib
import logging
import time
from dataclasses import dataclass, field
from typing import Optional

import httpx
import nest_asyncio
import trafilatura
from ddgs import DDGS
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder

# -- allow nested event loop for Jupyter / IPython compatibility -------------
nest_asyncio.apply()

# -- optional: swap for newspaper3k / BeautifulSoup as fallback --------------
try:
    from newspaper import Article as NewspaperArticle

    HAS_NEWSPAPER = True
except ImportError:
    HAS_NEWSPAPER = False


# -- logging setup -----------------------------------------------------------
log = logging.getLogger(__name__)

# ===============================================================================
# CONFIG
# ===============================================================================


@dataclass
class RetrievalConfig:
    # Search
    max_ddg_results: int = 7
    timeout_ddg: int = 10
    num_query_expansions: int = 1  # extra query variants (+ original)

    # Fetching
    fetch_timeout: float = 8.0
    max_concurrent_fetches: int = 6
    max_text_chars: int = 20_000

    # Chunking
    chunk_size: int = 450
    chunk_overlap: int = 80

    # Retrieval
    bm25_top_k: int = 10
    dense_top_k: int = 10
    rrf_k: int = 60  # RRF constant (60 is standard)
    diversity_max_per_url: int = 2

    # Reranking
    cross_encoder_model: str = "BAAI/bge-reranker-base"
    final_top_k: int = 3

    # LLM (query expansion)
    llm_model: str = "qwen2.5:7b-instruct-q4_K_M"
    embedding_model: str = "nomic-embed-text"


# ===============================================================================
# 1. QUERY EXPANSION
# ===============================================================================

_EXPANSION_PROMPT = """\
You are a search-query optimizer. Given the user query below, generate {n} \
alternative search queries that cover different phrasings, synonyms, or \
sub-aspects of the same information need.

Rules:
- Each alternative must be semantically distinct from the others.
- Keep each query concise (less than 12 words).
- Output ONLY the queries, one per line, no numbering, no explanation.

User query: {query}
"""


def expand_query(query: str, cfg: RetrievalConfig) -> list[str]:
    """Return original query + N LLM-generated variants."""
    if cfg.num_query_expansions == 0:
        return [query]

    llm = ChatOllama(model=cfg.llm_model, temperature=0)

    prompt = _EXPANSION_PROMPT.format(n=cfg.num_query_expansions, query=query)

    try:
        response = llm.invoke(prompt)
        variants = [
            line.strip()
            for line in response.content.strip().splitlines()
            if line.strip()
        ][: cfg.num_query_expansions]
    except Exception as exc:
        log.warning("Query expansion failed, falling back to original: %s", exc)
        variants = []

    all_queries = [query] + variants
    log.info("Expanded queries: %s", all_queries)
    return all_queries


# ===============================================================================
# 2. SEARCH & DEDUPLICATION
# ===============================================================================


def ddg_search(query: str, cfg: RetrievalConfig) -> list[dict]:
    with DDGS(timeout=cfg.timeout_ddg) as ddgs:
        return list(
            ddgs.text(
                query,
                max_results=cfg.max_ddg_results,
                backend="yandex, mojeek, startpage, yahoo",
            )
        )


def search_all_queries(queries: list[str], cfg: RetrievalConfig) -> list[dict]:
    """Search every expanded query and merge, deduplicating by URL."""
    seen_urls: set[str] = set()
    merged: list[dict] = []

    for q in queries:
        try:
            results = ddg_search(q, cfg)
        except Exception as exc:
            log.warning("DDG search failed for %r: %s", q, exc)
            continue

        for r in results:
            url = (r.get("href") or "").strip().lower().rstrip("/")
            if not url or url in seen_urls:
                continue
            seen_urls.add(url)
            merged.append(r)

    log.info("Total unique URLs after merging expansions: %d", len(merged))
    return merged


# ===============================================================================
# 3. ASYNC FETCHING
# ===============================================================================

_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; RAGBot/1.0)"}

_BLOCKED_EXTENSIONS = {".pdf", ".doc", ".docx", ".ppt", ".pptx", ".xls", ".xlsx"}
_BLOCKED_DOMAINS = {
    "twitter.com",
    "x.com",
    "instagram.com",
    "facebook.com",
    "tiktok.com",
    "reddit.com",  # mostly JS-rendered, low text yield
}


def _is_fetchable(url: str) -> bool:
    from urllib.parse import urlparse

    parsed = urlparse(url.lower())
    if any(parsed.netloc.endswith(d) for d in _BLOCKED_DOMAINS):
        return False
    if any(parsed.path.endswith(ext) for ext in _BLOCKED_EXTENSIONS):
        return False
    return True


def _extract_text(html: str) -> str:
    """Trafilatura -> newspaper3k fallback."""
    text = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=False,
        favor_recall=True,
    )
    if text and len(text) >= 200:
        return text

    if HAS_NEWSPAPER:
        try:
            article = NewspaperArticle(url="")
            article.set_html(html)
            article.parse()
            if article.text and len(article.text) >= 200:
                return article.text
        except Exception:
            pass

    return ""


async def _fetch_one(
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
    result: dict,
    cfg: RetrievalConfig,
) -> list[Document]:
    """Fetch a single URL and return chunked Documents."""
    url: str = result.get("href", "")
    title: str = result.get("title", "No title")
    snippet: str = result.get("body", "")

    if not url or not _is_fetchable(url):
        # Graceful fallback: use snippet only
        return (
            [
                Document(
                    page_content=snippet,
                    metadata={
                        "title": title,
                        "url": url,
                        "snippet": snippet,
                        "chunk_id": 0,
                    },
                )
            ]
            if snippet
            else []
        )

    async with semaphore:
        try:
            resp = await client.get(url, headers=_HEADERS, timeout=cfg.fetch_timeout)
            resp.raise_for_status()
            html = resp.text
        except Exception as exc:
            log.debug("Fetch failed for %s: %s", url, exc)
            return (
                [
                    Document(
                        page_content=snippet,
                        metadata={
                            "title": title,
                            "url": url,
                            "snippet": snippet,
                            "chunk_id": 0,
                        },
                    )
                ]
                if snippet
                else []
            )

    text = _extract_text(html)
    if not text:
        text = snippet

    text = " ".join(text.split())[: cfg.max_text_chars]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg.chunk_size,
        chunk_overlap=cfg.chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_text(text)

    return [
        Document(
            page_content=chunk,
            metadata={"title": title, "url": url, "snippet": snippet, "chunk_id": i},
        )
        for i, chunk in enumerate(chunks)
    ]


async def fetch_all_async(results: list[dict], cfg: RetrievalConfig) -> list[Document]:
    """Fetch all URLs concurrently, respecting max_concurrent_fetches."""
    semaphore = asyncio.Semaphore(cfg.max_concurrent_fetches)
    async with httpx.AsyncClient(follow_redirects=True) as client:
        tasks = [_fetch_one(client, semaphore, r, cfg) for r in results]
        batches = await asyncio.gather(*tasks, return_exceptions=True)

    docs: list[Document] = []
    for batch in batches:
        if isinstance(batch, Exception):
            log.warning("Fetch task raised: %s", batch)
            continue
        docs.extend(batch)

    log.info("Total chunks after async fetch: %d", len(docs))
    return docs


def fetch_all(results: list[dict], cfg: RetrievalConfig) -> list[Document]:
    """Sync entry-point compatible with Jupyter + scripts."""
    loop = asyncio.get_event_loop()
    if loop.is_running():
        # Jupyter / IPython case
        return loop.run_until_complete(fetch_all_async(results, cfg))
    else:
        # normal script case
        return asyncio.run(fetch_all_async(results, cfg))


# ===============================================================================
# DIVERSITY FILTER
# ===============================================================================


def diversity_filter(docs: list[Document], max_per_url: int = 2) -> list[Document]:
    seen: dict[str, int] = {}
    filtered = []
    for doc in docs:
        url = doc.metadata.get("url", "")
        if seen.get(url, 0) >= max_per_url:
            continue
        filtered.append(doc)
        seen[url] = seen.get(url, 0) + 1
    return filtered


# ===============================================================================
# 4. HYBRID RETRIEVAL: BM25 + DENSE + RRF
# ===============================================================================


def _rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank)


def hybrid_retrieve_rrf(
    query: str,
    docs: list[Document],
    cfg: RetrievalConfig,
) -> list[Document]:
    """
    1. BM25 retrieval   -> ranked list A
    2. Dense retrieval  -> ranked list B
    3. Fuse with RRF    -> final merged ranking
    """
    if not docs:
        return []

    # -- BM25 -----------------------------------------------------------------
    bm25_retriever = BM25Retriever.from_documents(docs, k=cfg.bm25_top_k)
    bm25_results: list[Document] = bm25_retriever.invoke(query)

    # -- Dense (FAISS + embeddings) -------------------------------------
    embeddings = OllamaEmbeddings(model=cfg.embedding_model)
    try:
        vectorstore = FAISS.from_documents(docs, embeddings)
        dense_results: list[Document] = vectorstore.similarity_search(
            query, k=cfg.dense_top_k
        )
    except Exception as exc:
        log.warning("Dense retrieval failed, using BM25 only: %s", exc)
        dense_results = []

    # -- RRF Fusion ------------------------------------------------------------
    def _doc_key(doc: Document) -> str:
        """Stable identity for a chunk (URL + chunk_id)."""
        return "{}::{}".format(
            doc.metadata.get("url", ""),
            doc.metadata.get(
                "chunk_id", hashlib.md5(doc.page_content.encode()).hexdigest()[:8]
            ),
        )

    scores: dict[str, float] = {}
    doc_map: dict[str, Document] = {}

    for rank, doc in enumerate(bm25_results):
        key = _doc_key(doc)
        scores[key] = scores.get(key, 0.0) + _rrf_score(rank, cfg.rrf_k)
        doc_map[key] = doc

    for rank, doc in enumerate(dense_results):
        key = _doc_key(doc)
        scores[key] = scores.get(key, 0.0) + _rrf_score(rank, cfg.rrf_k)
        doc_map[key] = doc

    ranked_keys = sorted(scores, key=scores.__getitem__, reverse=True)
    fused = []
    for k in ranked_keys:
        doc = doc_map[k]
        doc.metadata["rrf_score"] = scores[k]
        fused.append(doc)

    log.info(
        "RRF fusion: BM25=%d, dense=%d -> merged=%d",
        len(bm25_results),
        len(dense_results),
        len(fused),
    )
    return fused


# ===============================================================================
# 5. CROSS-ENCODER RERANKING
# ===============================================================================

_cross_encoder_cache: dict[str, CrossEncoder] = {}


def _get_cross_encoder(model_name: str) -> CrossEncoder:
    if model_name not in _cross_encoder_cache:
        log.info("Loading cross-encoder model: %s", model_name)
        _cross_encoder_cache[model_name] = CrossEncoder(model_name)
    return _cross_encoder_cache[model_name]


def cross_encoder_rerank(
    query: str,
    docs: list[Document],
    cfg: RetrievalConfig,
) -> list[Document]:
    if not docs:
        return []

    encoder = _get_cross_encoder(cfg.cross_encoder_model)
    pairs = [(query, doc.page_content) for doc in docs]

    try:
        scores = encoder.predict(pairs)
    except Exception as exc:
        log.warning("Cross-encoder failed, returning unranked docs: %s", exc)
        return docs[: cfg.final_top_k]

    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    top_docs = [doc for _, doc in ranked[: cfg.final_top_k]]

    log.info("Cross-encoder reranked %d -> %d docs", len(docs), len(top_docs))
    return top_docs


# ===============================================================================
# FORMATTING
# ===============================================================================


def format_evidence(docs: list[Document]) -> str:
    if not docs:
        return "No results found."

    blocks = []
    for i, doc in enumerate(docs, 1):
        blocks.append(
            f"[Chunk {i}]\n"
            f"TITLE: {doc.metadata.get('title')}\n"
            f"URL:   {doc.metadata.get('url')}\n"
            f"CONTENT:  {doc.page_content}\n"
            f"SNIPPET: {doc.metadata.get('snippet')}"
        )
    return "\n\n".join(blocks)


# ===============================================================================
# PUBLIC API FUNCTION
# ===============================================================================


def retrieve(query: str, cfg: Optional[RetrievalConfig] = None) -> str:
    """
    Full pipeline:
      query -> expand -> search -> async fetch -> diversity filter
            -> hybrid RRF retrieval -> cross-encoder rerank -> format
    """
    cfg = cfg or RetrievalConfig()
    t0 = time.perf_counter()

    # 1. Query expansion
    queries = expand_query(query, cfg)

    # 2. Search all variants, deduplicate by URL
    raw_results = search_all_queries(queries, cfg)
    if not raw_results:
        return "No results found."

    # 3. Async fetch + chunk
    docs = fetch_all(raw_results, cfg)

    # 4. Hybrid BM25 + dense + RRF
    fused_docs = hybrid_retrieve_rrf(query, docs, cfg)

    if not fused_docs:
        return "No results found."

    # 5. Cross-encoder rerank
    reranked_docs = cross_encoder_rerank(query, fused_docs, cfg)

    # 6. Per-URL diversity cap
    top_docs = diversity_filter(reranked_docs, max_per_url=cfg.diversity_max_per_url)

    elapsed = time.perf_counter() - t0
    log.info(
        "retrieve() completed in %.2fs, returning %d chunks", elapsed, len(top_docs)
    )

    return format_evidence(top_docs)


# ============================
# EXAMPLES
# ===========================

recent_question = Question(
    id=3,
    text="Who won the UEFA Champions League in 2025?",
    options=[
        AnswerOption(0, "Real Madrid"),
        AnswerOption(1, "Manchester City"),
        AnswerOption(2, "PSG"),
        AnswerOption(3, "Bayern Munich"),
    ],
)

evidence = retrieve(recent_question.text)
print(evidence)

2026-05-27 19:44:53,222 INFO httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 19:44:55,785 INFO __main__: Expanded queries: ['Who won the UEFA Champions League in 2025?', 'What is the UEFA Champions League winner for 2025?']
2026-05-27 19:44:58,267 INFO __main__: Total unique URLs after merging expansions: 12
2026-05-27 19:44:58,484 INFO httpx: HTTP Request: GET https://en.wikipedia.org/wiki/2025-26_UEFA_Champions_League "HTTP/1.1 200 OK"
2026-05-27 19:44:58,488 INFO httpx: HTTP Request: GET https://en.wikipedia.org/wiki/2022-23_UEFA_Champions_League "HTTP/1.1 200 OK"
2026-05-27 19:44:58,493 INFO httpx: HTTP Request: GET https://en.wikipedia.org/wiki/2023-24_UEFA_Champions_League "HTTP/1.1 200 OK"
2026-05-27 19:44:58,502 INFO httpx: HTTP Request: GET https://en.wikipedia.org/wiki/2024%E2%80%9325_UEFA_Champions_League "HTTP/1.1 200 OK"
2026-05-27 19:44:58,506 INFO httpx: HTTP Request: GET https://en.wikipedia.org/wiki/2022-23_Champions_League "HTTP/

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6026.60it/s]


2026-05-27 19:45:24,080 INFO httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-27 19:45:24,206 INFO httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-27 19:45:24,334 INFO httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-27 19:45:24,460 INFO httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-27 19:45:24,589 INFO httpx: HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-27 19:45:24,598 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/toke

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.83it/s]


2026-05-27 19:45:40,278 INFO __main__: Cross-encoder reranked 20 -> 3 docs
2026-05-27 19:45:40,284 INFO __main__: retrieve() completed in 47.82s, returning 3 chunks
[Chunk 1]
TITLE: 2025 UEFA Champions League final - Wikipedia
URL:   https://en.wikipedia.org/wiki/2025_UEFA_Champions_League_Final
CONTENT:  from outside of Europe's four big leagues (England, Spain, Italy, and Germany) had won the tournament
SNIPPET: The 2025 UEFA Champions League final will be the final match of the 2024–25 UEFA Champions League , the 70th season of Europe's premier club ...

[Chunk 2]
TITLE: 2022–23 UEFA Champions League - Wikipedia
URL:   https://en.wikipedia.org/wiki/2022-23_UEFA_Champions_League
CONTENT:  . Real Madrid were the defending champions, having won a record-extending fourteenth European Cup title in the previous edition, but they were eliminated by eventual champions Manchester City in the semi-finals
SNIPPET: A total of 78 teams from 53 of the 55 UEFA member associations participated in t

In [ ]:
from abc import ABC, abstractmethod
from typing import Any, Protocol

from src.polimillionaire.strategies import parse_answer_prediction


class BaseStrategy(ABC):
    name = "base"

    @abstractmethod
    def answer(self, question: Question) -> AnswerPrediction:
        raise NotImplementedError


class LocalLLM(Protocol):
    model_name: str

    def generate(self, prompt: str, **kwargs: object) -> str: ...


class OllamaStrategy(BaseStrategy):
    name = "Ollama-Gwen2.5-7b"

    def __init__(self):
        self.llm = ChatOllama(model="gwen2.5:7b", temperature=0)

    def answer(self, question: Question) -> AnswerPrediction:
        prediction = ask_question(question)

        # raw_text = self.llm.generate(build_prompt(question))
        prediction = parse_answer_prediction(
            str(prediction), question, strategy_name=self.name
        )
        prediction.metadata["strategy"] = self.name
        prediction.metadata["model_name"] = getattr(self.llm, "model_name", "unknown")
        prediction.metadata["device"] = getattr(self.llm, "device_summary", "unknown")
        return prediction

In [ ]:
from datetime import datetime, timezone

from polimillionaire.runner import GameRunner, RunLogger
from polimillionaire.strategies import GemmaLLMConfig, GemmaStrategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
)


def clear_model_memory():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if (
            hasattr(torch, "mps")
            and hasattr(torch.backends, "mps")
            and torch.backends.mps.is_available()
        ):
            torch.mps.empty_cache()
    except Exception as exc:
        print("Cleanup warning:", exc)


def make_strategy(model_id: str) -> GemmaStrategy:
    config = GemmaLLMConfig(
        model_id=model_id,
        inference_backend="auto_model",
        device_map="auto",
        dtype="auto",
        max_new_tokens=8,
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        seed=42,
        generation_max_time_seconds=18.0,
        timeout_seconds=120.0,
    )
    return GemmaStrategy(model_config=config)


def clean_name(label: str) -> str:
    return "_".join(label.lower().split())


if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue

    strategy = None
    try:
        print()
        print("Model:", item["label"])
        # strategy = make_strategy(item["model_id"])
        strategy = OllamaStrategy()

        warmup = strategy.answer(warmup_question)

        print("warmup option:", warmup.option_id, warmup.answer_text)
        print("fallback:", warmup.metadata.get("fallback"))
        print("device:", warmup.metadata.get("device"))
        if warmup.metadata.get("fallback"):
            raise RuntimeError(f"Warm-up failed for {item['label']}.")

        result = {
            "label": item["label"],
            "model_id": item["model_id"],
            "warmup_option": warmup.option_id,
        }

        if RUN_LIVE_GAME:
            run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            log_path = (
                REPO_ROOT
                / "results"
                / "runs"
                / f"{clean_name(item['label'])}_{run_id}.jsonl"
            )
            runner = GameRunner(
                client=client,
                safe_delay_seconds=SAFE_DELAY_SECONDS,
                answer_timeout_seconds=ANSWER_TIMEOUT_SECONDS,
                logger=RunLogger(log_path),
            )
            game = runner.play(COMPETITION_ID, strategy)
            result.update(
                {
                    "current_level": game.current_level,
                    "earned_amount": game.earned_amount,
                    "log_path": str(log_path),
                }
            )
            print("Reached level:", game.current_level)
            print("Earned amount:", game.earned_amount)
            print("Log path:", log_path)
        else:
            print("Live game skipped for", item["label"])

        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()
        print("Cleared model memory after", item["label"])

run_results


## Results

Read recent game logs.


In [ ]:
from polimillionaire.runner import load_jsonl, summarize_attempts

run_logs = sorted(
    (REPO_ROOT / "results" / "runs").glob("*.jsonl"),
    key=lambda path: path.stat().st_mtime,
)

for path in run_logs[-5:]:
    print(path.name)

if run_logs:
    rows = load_jsonl(run_logs[-1])
    print("latest:", run_logs[-1].name)
    print(summarize_attempts(rows))
else:
    print("No logs found.")


## Done

Use `run_results` and the JSONL logs for comparison.
